<a href="https://colab.research.google.com/github/diwashsapkota/EMDC-Training-Contents/blob/main/P2_transcript_to_derivative_content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Standalone Colab Pipeline
Transcript -> Derivative (schema) -> Article/Blog/Vlog

## Input Data

This notebook expects a transcript file as input. Supported formats include `.srt` (SubRip Subtitle) and `.txt` (plain text).

If the uploaded file is `.srt` or a `.txt` file with timestamps, the notebook will parse the timestamps and segment the text accordingly. If it's a plain `.txt` file without timestamps, the entire content will be treated as a single block of text.

In [ ]:
# @title Install dependencies
!pip -q install openai python-docx reportlab

import os
from getpass import getpass

try:
    from google.colab import userdata
    saved_key = userdata.get('OPENAI_API_KEY')
except Exception:
    saved_key = None

OPENAI_API_KEY = saved_key or getpass('Paste your OpenAI API key: ')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

if not os.environ['OPENAI_API_KEY'].startswith('sk-'):
    raise ValueError('OPENAI_API_KEY does not look valid.')

print('API key is ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.2 MB/s eta 0:00:00
Paste your OpenAI API key: ··········
API key is ready.


In [ ]:
# @title Setup and Confirguration

from google.colab import files
import re

print("Please upload your transcript file (e.g., .srt, .txt): ")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Please upload a transcript file.")

# Get the filename of the uploaded transcript
uploaded_filename = list(uploaded.keys())[0]
TRANSCRIPT_PATH = f'/content/{uploaded_filename}'

# Parameters that the user wants to keep or define
VIDEO_TITLE = 'Test_Project' # @param {"type":"string"}
OUT_DIR='EMDC Session' # @param {"type":"string"}

# Set YouTube-related variables to empty/default since it's a direct upload
YOUTUBE_URL = ''
# Derive VIDEO_ID from the uploaded filename for unique output directory and file names
# Sanitize the filename to make it a valid ID
VIDEO_ID = re.sub(r'[^a-zA-Z0-9_-]', '_', uploaded_filename.split('.')[0])

OUT_ROOT = f'/content/{OUT_DIR}/{VIDEO_ID}'

print(f"Transcript will be processed from: {TRANSCRIPT_PATH}")
print(f"Output will be saved to: {OUT_ROOT}")


Please upload your transcript file (e.g., .srt, .txt): 


Saving 20210408_PTXPrintItAlmostMiracle.srt to 20210408_PTXPrintItAlmostMiracle.srt
Transcript will be processed from: /content/20210408_PTXPrintItAlmostMiracle.srt
Output will be saved to: /content/EMDC Session/20210408_PTXPrintItAlmostMiracle


In [ ]:
# @title Cleaning

import json, re
from pathlib import Path
from openai import OpenAI

# Confirm paths after previous modifications
print(f"Processing transcript from: {TRANSCRIPT_PATH}")
print(f"Saving cleaned files to: {OUT_ROOT}")

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)

SRT_BLOCK_RE = re.compile(
    r'(?:^|\n)\s*(\d+)\s*\n'
    r'\s*(\d{2}:\d{2}:\d{2}[,. ]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[,. ]\d{3})\s*\n'
    r'(.*?)(?=\n\s*\d+\s*\n\s*\d{2}:\d{2}:\d{2}[,. ]\d{3}\s*-->|\Z)',
    re.S,
)
LINE_TS_RE = re.compile(r'^\s*(?:[?])?(?P<ts>(?:\d{1,2}:)?\d{1,2}:\d{2})(?:])?\s+(?P<text>.+?)\s*$')

def normalize_ts(ts: str) -> str:
    ts = ts.strip().replace(',', '.')
    if '.' not in ts: ts += '.000'
    hhmmss, ms = ts.split('.', 1)
    ms = (ms + '000')[:3]
    p = hhmmss.split(':')
    if len(p) == 2: hh, mm, ss = '00', p[0], p[1]
    else: hh, mm, ss = p
    return f'{int(hh):02d}:{int(mm):02d}:{int(ss):02d}.{ms}'

def parse_transcript(path: str):
    content = Path(path).read_text(encoding='utf-8', errors='ignore')

    raw_segments = []
    for m in SRT_BLOCK_RE.finditer(content):
        text = re.sub(r'\s+', ' ', m.group(4)).strip()
        if text:
            raw_segments.append({'index': len(raw_segments)+1, 'start': normalize_ts(m.group(2)), 'end': normalize_ts(m.group(3)), 'text': text})

    if not raw_segments:
        raw = []
        for line in content.splitlines():
            mm = LINE_TS_RE.match(line.strip())
            if mm: raw.append((normalize_ts(mm.group('ts')), mm.group('text').strip()))
        for i, (start, text) in enumerate(raw):
            end = raw[i+1][0] if i+1 < len(raw) else start
            raw_segments.append({'index': i+1, 'start': start, 'end': end, 'text': text})

    # If no timestamps are detected after both attempts, return as plain text
    if not raw_segments:
        return {'segments': [], 'full_text': content.strip()}

    # --- New logic to clean duplicate consecutive segments ---
    cleaned_segments = []
    if raw_segments:
        cleaned_segments.append(raw_segments[0]) # Add the first segment
        for i in range(1, len(raw_segments)):
            # Only add segment if its text is different from the previous one
            if raw_segments[i]['text'] != raw_segments[i-1]['text']:
                cleaned_segments.append(raw_segments[i])

    # Use the cleaned segments for further processing
    return {'segments': cleaned_segments, 'full_text': ' '.join(s['text'] for s in cleaned_segments).strip()}

parsed = parse_transcript(TRANSCRIPT_PATH)
Path(f'{OUT_ROOT}/transcript_clean.txt').write_text(parsed['full_text'], encoding='utf-8')

# Only write transcript_clean.srt if the original file was not a plain .txt
if not TRANSCRIPT_PATH.lower().endswith('.txt'):
    Path(f'{OUT_ROOT}/transcript_clean.srt').write_text(Path(TRANSCRIPT_PATH).read_text(encoding='utf-8', errors='ignore'), encoding='utf-8')

print('segments:', len(parsed['segments']))

Processing transcript from: /content/20210408_PTXPrintItAlmostMiracle.srt
Saving cleaned files to: /content/EMDC Session/20210408_PTXPrintItAlmostMiracle
segments: 1057


In [ ]:
# @title Paste yours JSON schema
DERIVATIVE_SCHEMA_JSON = r'''
{
  "name": "seminar_knowledge_derivative_schema",
  "strict": true,
  "schema": {
    "type": "object",
    "additionalProperties": false,
    "properties": {
      "about_video": {
        "type": "object",
        "additionalProperties": false,
        "properties": {
          "video_summary_short": {
            "type": "string"
          },
          "video_summary_long": {
            "type": "string"
          },
          "main_topic": {
            "type": "string"
          },
          "core_thesis": {
            "type": "string"
          },
          "speaker_perspective": {
            "type": "string"
          },
          "audience_relevance": {
            "type": "string"
          },
          "primary_purpose": {
            "type": "string"
          },
          "key_takeaways": {
            "type": "array",
            "items": {
              "type": "string"
            }
          },
          "major_themes": {
            "type": "array",
            "items": {
              "type": "string"
            }
          },
          "content_outline": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "title": {
                  "type": "string"
                },
                "summary": {
                  "type": "string"
                },
                "t1": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["title", "summary", "t1"]
            }
          },
          "mind_map": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "node": {
                  "type": "string"
                },
                "children": {
                  "type": "array",
                  "items": {
                    "type": "string"
                  }
                }
              },
              "required": ["node", "children"]
            }
          },
          "key_moments": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "moment_title": {
                  "type": "string"
                },
                "why_it_matters": {
                  "type": "string"
                },
                "t1": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["moment_title", "why_it_matters", "t1"]
            }
          },
          "important_quotes": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "quote": {
                  "type": "string"
                },
                "meaning": {
                  "type": "string"
                },
                "t1": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["quote", "meaning", "t1"]
            }
          },
          "glossary_core": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "term": {
                  "type": "string"
                },
                "definition": {
                  "type": "string"
                },
                "why_it_matters": {
                  "type": "string"
                }
              },
              "required": ["term", "definition", "why_it_matters"]
            }
          },
          "chapter_potential": {
            "type": "object",
            "additionalProperties": false,
            "properties": {
              "suggested_chapter_angle": {
                "type": "string"
              },
              "suggested_chapter_title": {
                "type": "string"
              },
              "chapter_position_in_book": {
                "type": "string"
              },
              "why_this_works_as_a_chapter": {
                "type": "string"
              }
            },
            "required": [
              "suggested_chapter_angle",
              "suggested_chapter_title",
              "chapter_position_in_book",
              "why_this_works_as_a_chapter"
            ]
          }
        },
        "required": [
          "video_summary_short",
          "video_summary_long",
          "main_topic",
          "core_thesis",
          "speaker_perspective",
          "audience_relevance",
          "primary_purpose",
          "key_takeaways",
          "major_themes",
          "content_outline",
          "mind_map",
          "key_moments",
          "important_quotes",
          "glossary_core",
          "chapter_potential"
        ]
      },
      "derivative": {
        "type": "object",
        "additionalProperties": false,
        "properties": {
          "alternative_titles": {
            "type": "array",
            "items": {
              "type": "string"
            }
          },
          "one_sentence_abstract": {
            "type": "string"
          },
          "social_captions": {
            "type": "array",
            "items": {
              "type": "string"
            }
          },
          "long_description": {
            "type": "string"
          },
          "summary": {
            "type": "string"
          },
          "lessons": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "point": {
                  "type": "string"
                },
                "explanation": {
                  "type": "string"
                },
                "t1": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["point", "explanation", "t1"]
            }
          },
          "discussion_questions": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "question": {
                  "type": "string"
                },
                "purpose": {
                  "type": "string"
                },
                "t1": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["question", "purpose", "t1"]
            }
          },
          "reflection_questions": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "question": {
                  "type": "string"
                },
                "focus": {
                  "type": "string"
                },
                "t1": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["question", "focus", "t1"]
            }
          },
          "keywords": {
            "type": "array",
            "items": {
              "type": "string"
            }
          },
          "glossary": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "term": {
                  "type": "string"
                },
                "definition": {
                  "type": "string"
                },
                "relevance": {
                  "type": "string"
                }
              },
              "required": ["term", "definition", "relevance"]
            }
          },
          "quotes": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "quote": {
                  "type": "string"
                },
                "meaning": {
                  "type": "string"
                },
                "t1": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["quote", "meaning", "t1"]
            }
          },
          "carousel": {
            "type": "object",
            "additionalProperties": false,
            "properties": {
              "theme": {
                "type": "string"
              },
              "slides": {
                "type": "array",
                "maxItems": 10,
                "items": {
                  "type": "object",
                  "additionalProperties": false,
                  "properties": {
                    "slide_number": {
                      "type": "integer"
                    },
                    "title": {
                      "type": "string"
                    },
                    "content": {
                      "type": "string"
                    },
                    "emotion": {
                      "type": "string"
                    }
                  },
                  "required": ["slide_number", "title", "content", "emotion"]
                }
              }
            },
            "required": ["theme", "slides"]
          },
          "clips": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "clip_title": {
                  "type": "string"
                },
                "reason": {
                  "type": "string"
                },
                "start_time": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2}$"
                },
                "end_time": {
                  "type": "string",
                  "pattern": "^\\d{2}:\\d{2}:\\d{2}$"
                }
              },
              "required": ["clip_title", "reason", "start_time", "end_time"]
            }
          },
          "visual_cards": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": false,
              "properties": {
                "card_title": {
                  "type": "string"
                },
                "card_theme": {
                  "type": "string"
                },
                "card_type": {
                  "type": "string"
                },
                "sections": {
                  "type": "array",
                  "items": {
                    "type": "object",
                    "additionalProperties": false,
                    "properties": {
                      "section_title": {
                        "type": "string"
                      },
                      "items": {
                        "type": "array",
                        "items": {
                          "type": "object",
                          "additionalProperties": false,
                          "properties": {
                            "title": {
                              "type": "string"
                            },
                            "content": {
                              "type": "string"
                            },
                            "highlight": {
                              "type": "string"
                            }
                          },
                          "required": ["title", "content", "highlight"]
                        }
                      }
                    },
                    "required": ["section_title", "items"]
                  }
                },
                "closing_message": {
                  "type": "string"
                }
              },
              "required": [
                "card_title",
                "card_theme",
                "card_type",
                "sections",
                "closing_message"
              ]
            }
          },
          "chapter": {
            "type": "object",
            "additionalProperties": false,
            "properties": {
              "chapter_title": {
                "type": "string"
              },
              "chapter_hook": {
                "type": "string"
              },
              "chapter_purpose": {
                "type": "string"
              },
              "target_reader_takeaway": {
                "type": "string"
              },
              "outline_chapter": {
                "type": "array",
                "items": {
                  "type": "object",
                  "additionalProperties": false,
                  "properties": {
                    "section_title": {
                      "type": "string"
                    },
                    "objective": {
                      "type": "string"
                    },
                    "key_points": {
                      "type": "array",
                      "items": {
                        "type": "string"
                      }
                    }
                  },
                  "required": ["section_title", "objective", "key_points"]
                }
              },
              "sections": {
                "type": "array",
                "items": {
                  "type": "object",
                  "additionalProperties": false,
                  "properties": {
                    "heading": {
                      "type": "string"
                    },
                    "purpose": {
                      "type": "string"
                    },
                    "content": {
                      "type": "string"
                    },
                    "source_range": {
                      "type": "string",
                      "pattern": "^\\d{2}:\\d{2}:\\d{2} - \\d{2}:\\d{2}:\\d{2}$"
                    }
                  },
                  "required": ["heading", "purpose", "content", "source_range"]
                }
              },
              "chapter_closing": {
                "type": "string"
              }
            },
            "required": [
              "chapter_title",
              "chapter_hook",
              "chapter_purpose",
              "target_reader_takeaway",
              "outline_chapter",
              "sections",
              "chapter_closing"
            ]
          }
        },
        "required": [
          "alternative_titles",
          "one_sentence_abstract",
          "social_captions",
          "long_description",
          "summary",
          "lessons",
          "discussion_questions",
          "reflection_questions",
          "keywords",
          "glossary",
          "quotes",
          "carousel",
          "clips",
          "visual_cards",
          "chapter"
        ]
      }
    },
    "required": ["about_video", "derivative"]
  }
}

'''


In [ ]:
# @title
derivative_schema = json.loads(DERIVATIVE_SCHEMA_JSON)
print('Schema loaded:', derivative_schema.get('name'))

assert derivative_schema['strict'] is True
assert 'schema' in derivative_schema and 'properties' in derivative_schema['schema']
assert 'about_video' in derivative_schema['schema']['properties']
assert 'derivative' in derivative_schema['schema']['properties']
print('Schema validation OK')

Schema loaded: seminar_knowledge_derivative_schema
Schema validation OK


In [ ]:
# @title Run

client = OpenAI()
system_der = 'You are an expert Christian content analyst. Create derivative English JSON from transcript only. No hallucination.'

# Conditionally construct user_der based on YOUTUBE_URL presence
if YOUTUBE_URL:
    user_der = f'''Analyze this transcript and return structured derivative JSON.

VIDEO_TITLE: {VIDEO_TITLE}
YOUTUBE_URL: {YOUTUBE_URL}
VIDEO_ID: {VIDEO_ID}

TRANSCRIPT:
{parsed['full_text']}'''
else:
    user_der = f'''Analyze this transcript and return structured derivative JSON.

TRANSCRIPT:
{parsed['full_text']}'''

resp = client.responses.create(
    model='gpt-5.4',
    input=[{'role':'system','content':system_der},{'role':'user','content':user_der}],
    text={'format': {'type':'json_schema','name': derivative_schema['name'],'strict': True,'schema': derivative_schema['schema']}}
)
derivative_json = json.loads(resp.output_text)

# Conditionally add sources if derivative exists and is from YouTube
if 'derivative' in derivative_json and isinstance(derivative_json['derivative'], dict):
    if YOUTUBE_URL: # Only add sources if YOUTUBE_URL is present
        derivative_json['derivative']['sources'] = {'youtube_url': YOUTUBE_URL, 'video_id': VIDEO_ID, 'title': VIDEO_TITLE}
Path(f'{OUT_ROOT}/derivative_en_{VIDEO_ID}.json').write_text(json.dumps(derivative_json, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved derivative_en json')


Saved derivative_en json


# Print Out

In [ ]:
# @title
import json
from IPython.display import HTML, display

def format_value_as_html(value, indent_level):
    indent_px = indent_level * 20 # Using pixels for indentation in HTML

    if isinstance(value, str):
        return value
    elif isinstance(value, list):
        if not value:
            return f"<p style='margin-left: {indent_px}px;'>No data available.</p>"
        list_items_html = []
        for item_idx, item in enumerate(value):
            item_content_html = ""
            if isinstance(item, dict):
                item_content_html = format_dict_as_html(item, indent_level + 1)
            elif isinstance(item, list):
                # For nested lists, simplify for now or recursively call format_value_as_html
                item_content_html = str(item)
            else:
                item_content_html = str(item)

            list_items_html.append(f"<li>{item_content_html}</li>") # List items naturally indent within <ul>

            if isinstance(item, dict) and item_idx < len(value) - 1:
                # Add a light horizontal rule between dictionary items in a list for clarity
                list_items_html.append(f"<hr style='margin-left: {indent_px}px; border-top: 1px solid #eee;'>")

        return f"<ul style='padding-left: {indent_px}px;'>{''.join(list_items_html)}</ul>" # Apply padding to the ul itself for base indent
    elif isinstance(value, dict):
        return format_dict_as_html(value, indent_level)
    elif value is None:
        return "N/A"
    else:
        return str(value)

def format_dict_as_html(data, indent_level):
    indent_px = indent_level * 20
    output_html_parts = []
    for key, value in data.items():
        formatted_key = key.replace('_', ' ').title()
        content_html = ""

        if isinstance(value, (str, int, float, bool)):
            content_html = f"<strong>{formatted_key}</strong>: {value}"
        elif isinstance(value, list):
            content_html = f"<strong>{formatted_key}</strong>:<br>" + format_value_as_html(value, indent_level + 1)
        elif isinstance(value, dict):
            content_html = f"<strong>{formatted_key}</strong>:<br>" + format_dict_as_html(value, indent_level + 1)
        elif value is None:
            content_html = f"<strong>{formatted_key}</strong>: N/A"
        else:
            content_html = f"<strong>{formatted_key}</strong>: {str(value)}"

        output_html_parts.append(f"<div style='margin-left: {indent_px}px; margin-bottom: 5px;'>{content_html}</div>")

    return ''.join(output_html_parts)

# Main logic to build the full HTML report
final_html_output = []
final_html_output.append("<h1>Output Derivative Data</h1>")

# Check if derivative_json exists in the environment
if 'derivative_json' in locals() and derivative_json:
    if 'about_video' in derivative_json:
        final_html_output.append("<h2>About Derivative</h2>")
        final_html_output.append(format_dict_as_html(derivative_json['about_video'], 1))

    if 'about_video' in derivative_json and 'derivative' in derivative_json:
        final_html_output.append("<hr style='border-top: 2px solid #ccc; margin: 20px 0;'>")

    if 'derivative' in derivative_json:
        final_html_output.append("<h2>Derivative Content</h2>")
        final_html_output.append(format_dict_as_html(derivative_json['derivative'], 1))
else:
    final_html_output.append("<p>'derivative_json' variable not found or empty. Please run the previous cell to generate it.</p>")

display(HTML("".join(final_html_output)))

# Download Output

In [ ]:
# @title
from google.colab import files

# Construct the full path to the derivative JSON file
derivative_file_path = f'{OUT_ROOT}/derivative_en_{VIDEO_ID}.json'

# Download the file
files.download(derivative_file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>